hi bye

In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

Mounted at /content/drive/


In [ ]:
import pandas as pd
import numpy as np
!pip install mne
import mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 68.9 MB/s eta 0:00:00


In [ ]:
#### Load data ####
raw_data = {"nabeha":[], "heidi":[], "avni":[]}

# Iterate thru all trials for each subject
for subj in raw_data.keys():
  for i in range(1,6):
    # Load data from CSV into an array
    trial_data = np.genfromtxt('/content/drive/Shareddrives/Neuromancers_Data/'+subj+'_data/OpenBCISession_'+subj+'_'+str(i)+'/BrainFlow-RAW_'+subj+'_'+str(i)+'_0.csv', delimiter='\t', dtype=str)
    trial_data = np.char.replace(trial_data, '\t', ' ')
    trial_data = trial_data.astype(float)

    # Declares channel names and types of each set of data
    ch_names = ['Channel {}'.format(i) for i in range(trial_data.shape[1])]
    ch_types = ['eeg' for i in range(trial_data.shape[1])]

    # Create info structures and RawArray objects for each set of data
    sfreq = 250  # sample rate in Hz
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    raw_array = mne.io.RawArray(trial_data.T, info)

    # Removing irrelevant channels
    ch_names = [raw_array.ch_names]
    ch_names_to_keep = [ch_names[0][0:10]]
    raw_array = raw_array.pick_channels(ch_names_to_keep[0])

    # Add RawArray
    raw_data[subj].append(raw_array)

Creating RawArray with float64 data, n_channels=24, n_times=42545
    Range : 0 ... 42544 =      0.000 ...   170.176 secs
Ready.
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Creating RawArray with float64 data, n_channels=24, n_times=46730
    Range : 0 ... 46729 =      0.000 ...   186.916 secs
Ready.
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Creating RawArray with float64 data, n_channels=24, n_times=45473
    Range : 0 ... 45472 =      0.000 ...   181.888 secs
Ready.
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Creating RawArray with float64 data, n_channels=24, n_times=42796
    Range : 0 ... 42795 =      0.000 ...   171.180 secs
Ready.
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Creating RawArray with float64 data, n_channels=24, n_times=40260
    Range : 0 ... 40259 =      0.000 ...   161.036 secs
Ready.
NOTE: pick_channels() is a legacy f

In [ ]:
import pandas as pd
import mne  # Assuming MNE is being used for EEG data

# Load segmentation data
data_segments = pd.read_csv('/content/drive/Shareddrives/Neuromancers_Data/EEG_Data_Segmentation.csv')

filtered_data = {"nabeha": [], "heidi": [], "avni": []}
baseline_data = {"nabeha": [], "heidi": [], "avni": []}

# Iterate through all trials for each subject
for subj in filtered_data.keys():
    for i in range(5):
        # Filter current trial data
        curr_trial = raw_data[subj][i]
        print(curr_trial)
        filtered_trial = curr_trial.copy().filter(
            l_freq=0.5, h_freq=45, picks=None, method='fir', fir_design='firwin',
            l_trans_bandwidth='auto', h_trans_bandwidth='auto',
            filter_length='auto', phase='zero'
        )

        # Removing Bad Channels
        bad_channels = ['Channel 0', 'Channel 9']  # List of channel names to mark as bad
        filtered_trial.info['bads'] = bad_channels
        filtered_trial.pick_types(eeg=True, exclude='bads')

        # Crop filtered_trial to within experiment duration
        curr_row = data_segments[data_segments["Participant"] == f"{subj}_{i+1}"].index
        start = data_segments.at[curr_row[0], "Experiment Start"]
        end = data_segments.at[curr_row[0], "Experiment End"]

        duration = filtered_trial.times[-1] - filtered_trial.times[0]  # Time in seconds
        if end > duration:
            end = duration -1

        print(f"{subj}_{i+1}: {start} {end}")

        baseline_trial = filtered_trial.copy()
        baseline_trial.crop(tmin=0, tmax=start)
        filtered_trial = filtered_trial.crop(tmin=start, tmax=end)

        # Add filtered_trial to filtered_data dictionary
        filtered_data[subj].append(filtered_trial)
        baseline_data[subj].append(baseline_trial)

<RawArray | 10 x 42545 (170.2 s), ~3.3 MiB, data loaded>
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 1651 samples (6.604 s)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
nabeha_1: 56.6031814 146.1235613
<RawArray | 10 x 46730 (186.9 s), ~3.6 MiB, data loaded>
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-d

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import numpy as np

X = []
y = []

# Feature extraction helper
def extract_features(raw):
    data = raw.get_data()  # shape: (n_channels, n_times)
    data = data[np.newaxis, ...]  # shape becomes (1, n_channels, n_times)
    mean = np.mean(data, axis=2)
    std = np.std(data, axis=2)
    max_val = np.max(data, axis=2)
    min_val = np.min(data, axis=2)
    features = np.concatenate((mean, std, max_val, min_val), axis=1).flatten()
    return features

# Loop through each subject's trials
for subj in filtered_data:
    for filtered_trial, baseline_trial in zip(filtered_data[subj], baseline_data[subj]):
        # Filtered data (during experiment)
        X.append(extract_features(filtered_trial))
        y.append(1) # Stress Signal
        # Baseline data (before experiment)
        X.append(extract_features(baseline_trial))
        y.append(0) # Baseline Signal

X = np.array(X)
y = np.array(y)

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
# SVM

from sklearn.svm import SVC

clf = SVC(kernel='linear', C=1)
clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"SVM Accuracy: {acc:.2f}")

SVM Accuracy: 0.83


In [ ]:
# LDA

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Train LDA
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)

# Predict and evaluate
y_pred = lda.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"LDA Accuracy: {acc:.2f}")

LDA Accuracy: 0.67


In [ ]:
# Random Forest

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy: {acc_rf:.2f}")

Random Forest Accuracy: 1.00


In [ ]:
# Neural Net

from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),  # 2 hidden layers with 64 and 32 neurons
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)
mlp.fit(X_train, y_train)
acc_mlp = accuracy_score(y_test, mlp.predict(X_test))
print(f"Neural Net (MLP) Accuracy: {acc_mlp:.2f}")

Neural Net (MLP) Accuracy: 1.00
